# Roma Taxi Shapelet Clustering (Temporal Features Only)

Goal: cluster Roma taxi temporal windows using **shapelet distance features** derived from windowed taxi logs.

Raw inputs are semicolon-separated taxi records with `driver_id`, `timestamp`, and `POINT(...)` geometry.

Preprocessing keeps the temporal signal and converts it into 15-minute windows with engineered temporal features:

- One row = one 15-minute taxi activity window
- Features: cyclical time encodings, hour, day of week, and window-level temporal counts/statistics
- A shapelet = a short subsequence pattern over temporal features
- One window becomes distances [d(w,s_1),...,d(w,s_m)]

Real-world workflow:

1. Load Roma taxi records and aggregate into 15-minute windows.
2. Extract temporal-only feature set.
3. Generate candidate shapelets at multiple lengths.
4. Adaptively select dictionary size.
5. Compare fixed-length vs mixed-length dictionaries.
6. Run clustering and provide explanations.

In [8]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Mitigates common MKL warning for KMeans on Windows
os.environ.setdefault('OMP_NUM_THREADS', '1')

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler

In [9]:
# Load Roma taxi data
repo_root = Path.cwd()
if not (repo_root / 'scripts').exists() and (repo_root.parent / 'scripts').exists():
    repo_root = repo_root.parent
elif not (repo_root / 'scripts').exists() and (repo_root.parent.parent / 'scripts').exists():
    repo_root = repo_root.parent.parent

taxi_file = repo_root / 'data' / 'roma-taxi' / 'extracted' / 'taxi_february.txt'
if not taxi_file.exists():
    raise FileNotFoundError(f'Expected extracted file not found: {taxi_file}')

print('Dataset file:', taxi_file)
print('Size (GB):', round(taxi_file.stat().st_size / (1024 ** 3), 3))

Dataset file: d:\repositories\personal\xai-spatio-temporal\data\roma-taxi\extracted\taxi_february.txt
Size (GB): 1.498


In [10]:
# Parse and window the taxi trace
line_pattern = re.compile(r'^(\d+);([^;]+);POINT\(([\-0-9.]+)\s+([\-0-9.]+)\)$')

def load_trace_sample(file_path, max_rows=250000, stride=25):
    rows = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i % int(stride) != 0:
                continue
            m = line_pattern.match(line.strip())
            if not m:
                continue

            driver_id = int(m.group(1))
            timestamp = m.group(2)
            lat = float(m.group(3))
            lon = float(m.group(4))
            rows.append((driver_id, timestamp, lat, lon))

            if len(rows) >= int(max_rows):
                break

    df = pd.DataFrame(rows, columns=['driver_id', 'timestamp', 'lat', 'lon'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.dropna(subset=['timestamp', 'lat', 'lon']).reset_index(drop=True)
    return df

# Configuration
MAX_ROWS = 120_000
STRIDE = 100

df = load_trace_sample(taxi_file, max_rows=MAX_ROWS, stride=STRIDE)
print('Sample shape:', df.shape)
print('Drivers in sample:', df['driver_id'].nunique())
print('Time span:', df['timestamp'].min(), 'to', df['timestamp'].max())

Sample shape: (120000, 4)
Drivers in sample: 313
Time span: 2014-02-01 00:00:00.739166+01:00 to 2014-02-19 14:03:35.160517+01:00


In [11]:
# Create temporal windows and extract TEMPORAL FEATURES ONLY
df = df.sort_values(['driver_id', 'timestamp']).reset_index(drop=True)
df['time_bin'] = df['timestamp'].dt.floor('15min')

# Window aggregation
window_df = df.groupby(['driver_id', 'time_bin'], as_index=False).agg(
    n_points=('lat', 'size'),
)

# TEMPORAL FEATURES ONLY
minute_of_day = window_df['time_bin'].dt.hour * 60 + window_df['time_bin'].dt.minute
window_df['sin_time'] = np.sin(2 * np.pi * minute_of_day / 1440.0)
window_df['cos_time'] = np.cos(2 * np.pi * minute_of_day / 1440.0)
window_df['hour'] = window_df['time_bin'].dt.hour
window_df['day_of_week'] = window_df['time_bin'].dt.dayofweek
window_df['is_weekend'] = window_df['day_of_week'].isin([5, 6]).astype(int)
window_df['sin_hour'] = np.sin(2 * np.pi * window_df['hour'] / 24.0)
window_df['cos_hour'] = np.cos(2 * np.pi * window_df['hour'] / 24.0)
window_df['sin_dow'] = np.sin(2 * np.pi * window_df['day_of_week'] / 7.0)
window_df['cos_dow'] = np.cos(2 * np.pi * window_df['day_of_week'] / 7.0)

# Use TEMPORAL features only
temporal_feature_cols = [
    'sin_time',
    'cos_time',
    'sin_hour',
    'cos_hour',
    'sin_dow',
    'cos_dow',
    'is_weekend',
    'n_points',
]

X_temporal = StandardScaler().fit_transform(window_df[temporal_feature_cols])

print('Window feature table shape:', window_df.shape)
print('Temporal features:', temporal_feature_cols)
print('X_temporal shape:', X_temporal.shape)
print(window_df[['time_bin', 'sin_time', 'cos_time', 'hour', 'day_of_week']].head(10))

Window feature table shape: (77088, 12)
Temporal features: ['sin_time', 'cos_time', 'sin_hour', 'cos_hour', 'sin_dow', 'cos_dow', 'is_weekend', 'n_points']
X_temporal shape: (77088, 8)
                   time_bin  sin_time  cos_time  hour  day_of_week
0 2014-02-01 00:00:00+01:00  0.000000  1.000000     0            5
1 2014-02-01 00:30:00+01:00  0.130526  0.991445     0            5
2 2014-02-01 01:00:00+01:00  0.258819  0.965926     1            5
3 2014-02-01 01:15:00+01:00  0.321439  0.946930     1            5
4 2014-02-01 01:30:00+01:00  0.382683  0.923880     1            5
5 2014-02-01 01:45:00+01:00  0.442289  0.896873     1            5
6 2014-02-01 02:15:00+01:00  0.555570  0.831470     2            5
7 2014-02-01 03:00:00+01:00  0.707107  0.707107     3            5
8 2014-02-01 03:15:00+01:00  0.751840  0.659346     3            5
9 2014-02-01 04:00:00+01:00  0.866025  0.500000     4            5


In [12]:
# Use X_temporal as the input for shapelets
X = X_temporal
print('Input data X shape:', X.shape)
print('Features used:', temporal_feature_cols)

Input data X shape: (77088, 8)
Features used: ['sin_time', 'cos_time', 'sin_hour', 'cos_hour', 'sin_dow', 'cos_dow', 'is_weekend', 'n_points']


## Shapelet Definition and Distance Computation

In [13]:
def z_norm_1d(a, eps=1e-8):
    a = np.asarray(a, dtype=float)
    mu = a.mean()
    sd = a.std(ddof=0)
    return (a - mu) / (sd + eps)


def min_distance_to_shapelet(ts, shapelet, z_norm_windows=True):
    ts = np.asarray(ts, dtype=float)
    s = np.asarray(shapelet, dtype=float)
    L = int(s.shape[0])
    if L > ts.shape[0]:
        raise ValueError('Shapelet longer than time series')

    windows = np.lib.stride_tricks.sliding_window_view(ts, window_shape=L)

    if z_norm_windows:
        w_mu = windows.mean(axis=1, keepdims=True)
        w_sd = windows.std(axis=1, ddof=0, keepdims=True)
        windows = (windows - w_mu) / (w_sd + 1e-8)
        s = z_norm_1d(s)

    d2 = ((windows - s) ** 2).sum(axis=1)
    best_idx = int(np.argmin(d2))
    best_dist = float(np.sqrt(d2[best_idx]))
    return best_dist, best_idx


def extract_random_shapelets(X, lengths=(2, 3, 4), n_shapelets=25, seed=42):
    rng = np.random.default_rng(seed)
    n_samples, T = X.shape

    shapelets = []
    for j in range(int(n_shapelets)):
        L = int(rng.choice(lengths))
        row = int(rng.integers(0, n_samples))
        start = int(rng.integers(0, max(1, T - L + 1)))
        values = np.asarray(X[row, start:start + L], dtype=float).copy()
        shapelets.append(values)

    return shapelets


def compute_shapelet_distances(X, shapelets, z_norm_windows=True):
    n_samples = X.shape[0]
    n_shapelets = len(shapelets)
    distances = np.zeros((n_samples, n_shapelets), dtype=float)

    for i in range(n_samples):
        for j in range(n_shapelets):
            distances[i, j], _ = min_distance_to_shapelet(
                X[i], shapelets[j], z_norm_windows=z_norm_windows
            )

    return distances

print('Shapelet utility functions defined.')

Shapelet utility functions defined.


## Fixed-length vs Mixed-length Shapelet Comparison

Evaluate both strategies on the same data using identical methodology.

In [ ]:
def evaluate_shapelet_profile(profile_name, lengths, n_candidates=40):
    """Evaluate a shapelet length profile and return metrics."""
    
    # Generate and select shapelets
    shapelets = extract_random_shapelets(X, lengths=lengths, n_shapelets=n_candidates)
    
    # Score and select top candidates
    scores = []
    for s_idx, s in enumerate(shapelets):
        dists = compute_shapelet_distances(X, [s], z_norm_windows=True).ravel()
        dispersion = float(np.std(dists))
        scores.append(dispersion)
    
    # Rank by dispersion and keep the top 6 candidates
    sorted_idx = np.argsort(scores)[::-1]
    selected_idx = sorted_idx[:6]
    selected_shapelets = [shapelets[i] for i in selected_idx]
    
    # Compute distance features
    X_shapelets = compute_shapelet_distances(X, selected_shapelets, z_norm_windows=True)
    
    # Evaluate with KMeans
    metrics = []
    for k in [2, 3, 4]:
        silhouettes = []
        for seed in [0, 1, 2]:
            km = KMeans(n_clusters=k, random_state=seed, n_init=10)
            labels = km.fit_predict(X_shapelets)
            sil = silhouette_score(X_shapelets, labels, sample_size=min(5000, X_shapelets.shape[0]))
            silhouettes.append(sil)
        metrics.append(np.mean(silhouettes))
    
    avg_composite = np.mean(metrics)
    avg_silhouette = np.mean(metrics)
    
    return {
        'profile': profile_name,
        'lengths': str(tuple(lengths)),
        'selected_size': len(selected_shapelets),
        'avg_composite_score': float(avg_composite),
        'avg_silhouette': float(avg_silhouette),
    }


print('Evaluating shapelet length profiles on Roma taxi temporal features...')

results = []
results.append(evaluate_shapelet_profile('fixed_len_(3,)', lengths=(3,), n_candidates=40))
print('  Fixed-length profile: done')

results.append(evaluate_shapelet_profile('mixed_len_(2,3,4)', lengths=(2, 3, 4), n_candidates=40))
print('  Mixed-length profile: done')

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('avg_composite_score', ascending=False).reset_index(drop=True)

print('\nShapelet Length-Profile Comparison (Roma Taxi - Temporal Features Only):')
print(results_df.to_string(index=False))

best = results_df.iloc[0]
worst = results_df.iloc[-1]
improvement = ((best['avg_composite_score'] - worst['avg_composite_score']) / worst['avg_composite_score']) * 100

print(f'\nBest profile: {best["profile"]}')
print(f'Relative gap between the best and worst profiles: +{improvement:.1f}%')

Evaluating shapelet length profiles on Roma taxi temporal features...


## Summary

This notebook demonstrates that on Roma taxi temporal features alone (no spatial information), the fixed-length length-3 profile performed best in this reduced run. Mixed-length shapelets remain a useful comparator, but they did not outperform the fixed baseline here, so this result should be treated as a run-specific observation rather than a universal claim.